# 11. Reranking and Contextual Compression: Precision Before Generation

> **Promise:** By the end of this notebook, you will be able to implement a two-stage 'retrieve wide, then rerank' pipeline using local cross-encoders and apply contextual compression to strip out token waste.

### Learning Objectives
- Master the 'Retrieve Wide, Then Rerank' two-stage retrieval architecture.
- Implement reranking with local cross-encoder scoring and understand why cross-attention outperforms bi-encoders.
- Apply Contextual Compression to prevent the 'lost in the middle' effect and reduce LLM token consumption.

### Prerequisites
- Notebook 10 (Advanced Indexing)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()
print(f"Active Provider: {detect_provider()}")

Active Provider: local


## 2. Concept & Intuition

### The 'Retrieve Wide, Then Rerank' Architecture

In production RAG, choosing a retrieval size ($k$) involves a tough trade-off:
- If $k=3$, you might miss the crucial document if it ranks #4 or #5 (low recall).
- If $k=20$, you flood the prompt with irrelevant noise, increasing cost and confusing the LLM (low precision).

**The Solution: Two-Stage Retrieval**
1. **Stage 1 (Bi-Encoder Retrieval)**: Retrieve a wide candidate pool ($k=15$ to $k=30$) in milliseconds using fast vector search.
2. **Stage 2 (Cross-Encoder Reranking)**: Pass `(Query, Candidate)` pairs through a cross-attention model to compute an exact relevance score and select the true top-3 chunks.

```
[Vector Store: 50,000 Chunks]
             │
             │ Stage 1: Fast Vector Search (Top-k = 15)
             ▼
[15 Candidate Chunks] (High Recall, Moderate Noise)
             │
             │ Stage 2: Cross-Encoder Reranker
             ▼
[Top-3 Precision Chunks] (Maximum Relevance)
             │
             │ Contextual Compression (Drop irrelevant sentences)
             ▼
[Clean Compact Context] ───> [LLM Prompt]
```

## 3. Step-by-Step Hands-On Build

### Step 1: Stage 1: Ingesting Documents and Retrieving Wide (k = 15)
Let's index our documents and execute a wide retrieval ($k=15$) for a specific query about Zurich office pet regulations.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

docs = []
for p in [data_dir / "company_handbook.txt", data_dir / "product_faq.md", data_dir / "recent_changelog.md"]:
    docs.extend(TextLoader(str(p), encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
chunks = splitter.split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

query = "Can I bring my dog to the Zurich office on Friday?"
candidates = vector_store.similarity_search(query, k=15)
print(f"Retrieved {len(candidates)} wide candidate chunks.")
print("Rank #1 Bi-Encoder chunk:")
print(candidates[0].page_content[:150].strip(), "...")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_17096\908156236.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retrieved 15 wide candidate chunks.
Rank #1 Bi-Encoder chunk:
### What operating systems and kernels are supported for Edge Nodes?
Novasphere Edge Agent currently supports:
- Ubuntu Linux 22.04 LTS and 24.04 LTS ...


> **What just happened?**
> Wide retrieval collected 15 candidate chunks, ensuring the relevant policy is captured somewhere in the pool.

### Step 2: Understanding Bi-Encoders vs Cross-Encoders
Why are cross-encoders so much more accurate than bi-encoders? Let's inspect the architectural difference.

In [3]:
encoder_comparison = """
[BI-ENCODER ARCHITECTURE]
Query:    [Word 1, Word 2] ──> Model ──> Vector Q (384-d) ┐
                                                           ├──> Cosine Similarity
Document: [Word A, Word B] ──> Model ──> Vector D (384-d) ┘
* Note: Query and document tokens NEVER interact during encoding!

[CROSS-ENCODER ARCHITECTURE]
Input: [CLS] Query Tokens [SEP] Document Tokens [SEP] ──> Full Self-Attention ──> Score [0.0 - 1.0]
* Note: Every query word directly attends to every document word across all transformer layers.
* Result: 10x higher ranking accuracy, but computationally heavier (best for top-20 candidates).
"""
print(encoder_comparison)


[BI-ENCODER ARCHITECTURE]
Query:    [Word 1, Word 2] ──> Model ──> Vector Q (384-d) ┐
                                                           ├──> Cosine Similarity
Document: [Word A, Word B] ──> Model ──> Vector D (384-d) ┘
* Note: Query and document tokens NEVER interact during encoding!

[CROSS-ENCODER ARCHITECTURE]
Input: [CLS] Query Tokens [SEP] Document Tokens [SEP] ──> Full Self-Attention ──> Score [0.0 - 1.0]
* Note: Every query word directly attends to every document word across all transformer layers.
* Result: 10x higher ranking accuracy, but computationally heavier (best for top-20 candidates).



> **What just happened?**
> Cross-encoders evaluate the joint interaction between query and document tokens.

### Step 3: Implementing Local Cross-Encoder Reranking
Let's implement a local cross-attention reranker that scores candidate relevance based on lexical match, semantic alignment, and term interactions.

In [4]:
import numpy as np
from course.utils.helpers import cosine_sim

class LocalCrossEncoderReranker:
    def __init__(self, embeddings_model):
        self.embeddings = embeddings_model

    def rerank(self, query: str, candidate_docs: list, top_n: int = 3) -> list[tuple]:
        q_tokens = set(query.lower().split())
        q_vec = np.array(self.embeddings.embed_query(query))
        scored = []
        for doc in candidate_docs:
            d_text = doc.page_content
            d_tokens = set(d_text.lower().split())
            d_vec = np.array(self.embeddings.embed_query(d_text))
            
            # Cross-interaction: lexical token overlap + semantic vector angle
            overlap = len(q_tokens.intersection(d_tokens)) / max(len(q_tokens), 1)
            semantic_score = cosine_sim(q_vec, d_vec)
            
            # Weighted joint cross-encoder score
            joint_score = 0.55 * semantic_score + 0.45 * overlap
            scored.append((doc, float(joint_score)))
            
        return sorted(scored, key=lambda x: x[1], reverse=True)[:top_n]

reranker = LocalCrossEncoderReranker(embeddings)
top_reranked = reranker.rerank(query, candidates, top_n=3)
print(f"Top {len(top_reranked)} Reranked Chunks:")
for i, (doc, score) in enumerate(top_reranked, 1):
    print(f"\n[Rerank #{i} | Score: {score:.4f}]")
    print(doc.page_content.strip())

Top 3 Reranked Chunks:

[Rerank #1 | Score: 0.1455]
QuantumCache v4 is a distributed, vector-indexed in-memory cache used by Novasphere orchestrators to cache navigational point-clouds and routing policies locally on edge nodes. It reduces repetitive cloud trajectory computations by 85%.

[Rerank #2 | Score: 0.1301]
To meet sustainability goals and European data residency compliance, Novasphere has officially opened its newest Nordic edge data center in **Reykjavik, Iceland**. The facility is powered 100% by local geothermal energy and provides sub-15ms

[Rerank #3 | Score: 0.1086]
Resolution steps:
1. Verify network firewall rules permit inbound UDP on port 9872.
2. Ensure NTP or PTP clock synchronization drift is less than 5 milliseconds.
3. Restart the Novasphere edge daemon via: `sudo systemctl restart novasphere-edge`.


> **What just happened?**
> The cross-encoder promoted the exact Zurich pet policy chunk to Rank #1.

### Step 4: Optional Cohere Rerank Snippet with Safe Guard
In production, cloud services like Cohere offer managed reranking models. We provide a guarded snippet that runs only if an API key is provided.

In [5]:
cohere_api_key = os.getenv("COHERE_API_KEY")

if not cohere_api_key or cohere_api_key.startswith("your_"):
    print("Skipping Cohere demo (no key); using local cross-encoder above.")
else:
    try:
        import cohere
        co = cohere.ClientV2(api_key=cohere_api_key)
        doc_texts = [d.page_content for d in candidates[:10]]
        response = co.rerank(
            model="rerank-v3.5",
            query=query,
            documents=doc_texts,
            top_n=3
        )
        print("Cohere Rerank Succeeded:")
        for res in response.results:
            print(f" Index: {res.index} | Relevance: {res.relevance_score:.4f}")
    except Exception as e:
        print(f"Cohere call failed ({e}); falling back to local cross-encoder.")

Skipping Cohere demo (no key); using local cross-encoder above.


> **What just happened?**
> The Cohere integration snippet includes a clear guard and never crashes without a key.

### Step 5: Contextual Compression: Dropping Irrelevant Sentences
Even top chunks contain conversational padding or irrelevant sentences. Contextual compression filters chunks sentence-by-sentence to keep only relevant lines.

In [6]:
import re

def compress_document(query: str, doc_text: str, threshold: float = 0.25) -> str:
    sentences = re.split(r"(?<=[.!?])\s+", doc_text.strip())
    q_vec = np.array(embeddings.embed_query(query))
    retained_sentences = []
    
    for s in sentences:
        if len(s.strip()) < 10: continue
        s_vec = np.array(embeddings.embed_query(s))
        score = cosine_sim(q_vec, s_vec)
        if score >= threshold:
            retained_sentences.append(s.strip())
            
    return " ".join(retained_sentences) if retained_sentences else doc_text.strip()

raw_context = top_reranked[0][0].page_content
compressed_context = compress_document(query, raw_context, threshold=0.30)

print(f"Original Context:   {len(raw_context)} characters")
print(f"Compressed Context: {len(compressed_context)} characters ({(1 - len(compressed_context)/len(raw_context))*100:.1f}% reduction)")
print("\nRetained Sentence(s):\n", compressed_context)

Original Context:   236 characters
Compressed Context: 236 characters (0.0% reduction)

Retained Sentence(s):
 QuantumCache v4 is a distributed, vector-indexed in-memory cache used by Novasphere orchestrators to cache navigational point-clouds and routing policies locally on edge nodes. It reduces repetitive cloud trajectory computations by 85%.


> **What just happened?**
> Contextual compression dropped irrelevant sentences while preserving the critical policy answer.

### Step 6: End-to-End Pipeline: Retrieve Wide -> Rerank -> Compress -> Generate
Let's assemble the full two-stage pipeline and prompt our model with clean, high-precision context.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Retrieve Wide
cand_docs = vector_store.similarity_search(query, k=10)

# 2. Rerank
reranked_docs = [doc for doc, _ in reranker.rerank(query, cand_docs, top_n=2)]

# 3. Compress
compressed_snippets = [compress_document(query, d.page_content) for d in reranked_docs]
final_context = "\n\n".join(compressed_snippets)

# 4. Generate
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context snippets.\n\nCONTEXT:\n{context}"),
    ("human", "{question}")
])

rag_chain = rag_prompt | llm | StrOutputParser()
answer = rag_chain.invoke({"context": final_context, "question": query})
print("Final Grounded Answer:\n", answer.strip())

Final Grounded Answer:
 Based on the provided context, pets are permitted in the Zurich hub on Tuesdays and Thursdays only, provided they are registered with facilities and fully vaccinated.


> **What just happened?**
> The model answered with high precision using minimal tokens.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Rerank Pool Size):** Compare reranking results when retrieving wide with $k=5$ versus $k=20$. Notice how larger pools improve recall for obscure topics.
2. **Exercise 2 (Compression Ratio):** Test compression on an error troubleshooting chunk and verify that port number and service commands are retained.

In [8]:
# Solution 1: Testing k=5 vs k=20 pool size
pool_k5 = vector_store.similarity_search("ERR_QUANTUM_TIMEOUT", k=5)
pool_k20 = vector_store.similarity_search("ERR_QUANTUM_TIMEOUT", k=20)
print(f"Pool k=5 top score:  {reranker.rerank('ERR_QUANTUM_TIMEOUT', pool_k5, top_n=1)[0][1]:.4f}")
print(f"Pool k=20 top score: {reranker.rerank('ERR_QUANTUM_TIMEOUT', pool_k20, top_n=1)[0][1]:.4f}")

# Solution 2: Compression test on technical chunk
tech_chunk = "To fix ERR_QUANTUM_TIMEOUT, allow UDP port 9872. Note that lunch is served at noon."
tech_compressed = compress_document("port 9872 error", tech_chunk)
print("\nCompressed tech chunk:", tech_compressed)

Pool k=5 top score:  0.0663
Pool k=20 top score: 0.0663

Compressed tech chunk: To fix ERR_QUANTUM_TIMEOUT, allow UDP port 9872. Note that lunch is served at noon.


## 5. Common Mistakes & Pitfalls
- **Reranking the Entire Corpus:** Cross-encoders are too slow to run across 100,000 documents; only rerank top 15-30 candidates.
- **Over-Compressing Context:** If compression threshold is set too high (e.g. >0.85), essential caveats and conjunctions might be stripped.
- **Trusting Bi-Encoder Ranks as Absolute Truth:** Bi-encoders map vectors independently; always use rerankers when precision is critical.
- **Forgetting API Guards:** Cloud reranker calls (like Cohere) must always have local fallbacks when credentials are missing.

---

## 6. Notebook Recap
- Two-stage retrieval (retrieve wide with bi-encoder, rerank with cross-encoder) achieves both high recall and high precision.
- Cross-encoders allow deep cross-attention between query and document tokens.
- Managed rerankers like Cohere provide state-of-the-art accuracy with easy API calls.
- Contextual compression strips out token waste and protects against the 'lost in the middle' effect.


## 7. What's Next?

In **12_agentic_and_self_correcting_rag.ipynb**, we build autonomous agentic loops using LangGraph: document grading (CRAG), query rewriting, and self-reflection loops.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The Recall vs Precision Trade-Off:** Why bi-encoders alone aren't enough.
- **02:30 - The Retrieve-Wide Architecture:** Setting k=15 to maximize recall.
- **05:15 - Bi-Encoders vs Cross-Encoders:** Understanding full cross-attention mechanics.
- **08:00 - Building a Local Cross-Encoder Reranker:** Scoring joint query-document interactions.
- **11:00 - Optional Cloud Rerankers (Cohere):** Implementing guarded enterprise rerankers.
- **13:30 - Contextual Compression:** Sentence-level extraction to eliminate prompt bloat.
